# 04 — SedonaDB nearest-road processing

**Learning objectives:** understand proved-radius candidate filtering,
prefilter the object relation correctly, calculate a closest point, and read
the plan.

`ST_KNN` is excellent for point neighbours, but Sedona documents centroid
semantics when an input is a non-point geometry. Roads are LineStrings. This
notebook therefore uses an exact `ST_DWithin` candidate join and distance
ranking. The tier is materialized before radius discovery and execution.

In [1]:
import os
from pathlib import Path

from sedona_benchmark.benchmark import _sedona_query
from sedona_benchmark.config import load_config

root = Path(os.environ.get("BENCHMARK_DATA_DIR", "/benchmark-data")) / "canonical"
roads_path = root / "israel_roads.parquet"
location_paths = sorted(root.glob("israel_locations_*.parquet"))
if not roads_path.exists() or not location_paths:
    raise FileNotFoundError("Prepare canonical roads and locations first")

config = load_config(os.environ.get("BENCHMARK_CONFIG", "config/benchmark.yaml"))
smoke = not (root / f"israel_locations_{config.values['locations']['count']}.parquet").exists()
sd, query, radius_distribution = _sedona_query(
    config, "general_driving", smoke=smoke
)
print(
    {
        "mode": "100-location smoke" if smoke else "10,000-location full",
        "radius_distribution": radius_distribution,
        "covered": sum(radius_distribution.values()),
    }
)

{'mode': '10,000-location full', 'radius_distribution': {1000: 5141, 2000: 1421, 4000: 1489, 8000: 1310, 16000: 639}, 'covered': 10000}


In [2]:
print(sd.sql(query).explain().to_pandas().to_string(index=False)[:8000])

    plan_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [3]:
sample = sd.sql(f"SELECT * FROM ({query}) WHERE location_id < 10").to_pandas(
    geometry="nearest_point"
)
sample[["location_id", "road_id", "road_class", "distance_m"]]

,location_id,road_id,road_class,distance_m
0,6,b16c7275-0bc2-44d1-b8e1-a3722ee31c64#0,unclassified,294.725380
1,8,bebbe276-0954-4444-a42a-4b2d86ce70b0#0,unclassified,44.130606
2,1,61a74f2e-8277-4c6b-82b3-adb11b7fb3b0#0,trunk,6519.810816
3,9,a6bd9f17-fae6-4d16-a83d-410db80725c3#0,residential,573.004087
4,3,1b125412-77e9-47fb-b399-8b193f1d28cc#0,trunk,633.238746
5,5,de36a7c0-6395-46e7-bbb4-5a35f2edd0a0#0,trunk,143.017235
6,7,573219a6-a027-45e7-8db2-ee94ae60581b#0,secondary,109.188928
7,4,6bfef759-ef9c-4095-8f95-6a1a89bc56b2#0,unclassified,18.720547
8,0,2d250a4b-ea9a-4e17-8793-e397125a7e83#0,primary,1045.604392
9,2,fe8c2c29-e170-4edd-a9db-2898942dd457#0,unclassified,13150.808580


Radius discovery is outside the timed query. Each location is assigned to the
first doubling radius that covers it, so sparse points do not force dense
points into a 16 km candidate join. The plan shows one literal-radius spatial
join per populated bucket, a union, and one global distance ranking.

The measured runner materializes every row and hashes the ordered result.
Timing `head()` would measure only enough work to return a display sample.